In [3]:
from flask import Flask, request, render_template_string
import pandas as pd
import joblib

app = Flask(__name__)
model = joblib.load("bank_model.pkl")

# (name, type, default or choices)
FIELDS = [
    ("age", "num", 40),
    ("job", "cat", ["admin.", "blue-collar", "entrepreneur", "housemaid", "management", "retired",
                    "self-employed", "services", "student", "technician", "unemployed", "unknown"]),
    ("marital", "cat", ["married", "single", "divorced"]),
    ("education", "cat", ["primary", "secondary", "tertiary", "unknown"]),
    ("balance", "num", 1000),
    ("housing", "cat", ["yes", "no"]),
    ("loan", "cat", ["yes", "no"]),
    ("contact", "cat", ["cellular", "telephone", "unknown"]),
    ("campaign", "num", 2),
    ("previous", "num", 0),
    ("poutcome", "cat", ["failure", "other", "success", "unknown"]),
]

PAGE = """
<!doctype html>
<title>Bank Marketing</title>
<style>
  body{font-family:Arial;max-width:600px;margin:30px auto;background:#f4f6f9}
  form{background:#fff;padding:20px;border-radius:8px}
  label{display:block;margin-top:10px;font-weight:bold}
  input,select{width:100%;padding:7px;box-sizing:border-box}
  button{margin-top:15px;padding:10px 20px;background:#2b6cb0;color:#fff;border:0;border-radius:6px}
  .res{background:#e6fffa;padding:15px;border-radius:8px;margin-top:15px}
</style>
<h2>Bank Marketing - Logistic Regression</h2>
<form method="post">
  {% for name, kind, opt in fields %}
    <label>{{ name }}</label>
    {% if kind == 'num' %}
      <input type="number" step="any" name="{{ name }}" value="{{ vals.get(name, opt) }}" required>
    {% else %}
      <select name="{{ name }}">
        {% for c in opt %}<option {{ 'selected' if vals.get(name) == c else '' }}>{{ c }}</option>{% endfor %}
      </select>
    {% endif %}
  {% endfor %}
  <button>Predict</button>
</form>
{% if result %}
<div class="res"><h3>Prediction: {{ result }}</h3>
  Will subscribe: {{ yes }}% | Will not subscribe: {{ no }}%</div>
{% endif %}
"""

@app.route("/", methods=["GET", "POST"])
def home():
    vals, result, yes, no = {}, None, 0, 0
    if request.method == "POST":
        vals = request.form.to_dict()
        row = {}
        for name, kind, _ in FIELDS:
            row[name] = float(vals[name]) if kind == "num" else vals[name]
        p = model.predict_proba(pd.DataFrame([row]))[0]
        no, yes = round(p[0] * 100, 2), round(p[1] * 100, 2)
        result = "YES - will subscribe" if p[1] > p[0] else "NO - will not subscribe"
    return render_template_string(PAGE, fields=FIELDS, vals=vals, result=result, yes=yes, no=no)

if __name__ == "__main__":
  app.run(debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Oct/2026 12:00:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 12:00:15] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Oct/2026 12:00:15] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [13/Oct/2026 12:00:30] "POST / HTTP/1.1" 200 -
